# ECON N171: Economic Development
## Lecture 4 — Pandas II: Grouping, Merging, Reshaping, and Cleaning
**Instructor: Rajveer Jat | Summer 2026**

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Use **groupby** to compute statistics within groups
2. **Merge** two datasets on a common key
3. **Reshape** data between wide and long formats
4. **Clean** messy real-world data
5. Build a **panel dataset** from World Bank data

> Run each cell with Shift + Enter and follow along.

In [1]:
import pandas as pd
import numpy as np

data = {
    'country'       : ['USA', 'France', 'Japan', 'UAE',
                       'Brazil', 'Mexico', 'South Korea',
                       'Vietnam', 'Nigeria', 'Ghana',
                       'Ethiopia', 'Kenya'],
    'region'        : ['North America', 'Europe', 'East Asia', 'Middle East',
                       'Latin America', 'Latin America', 'East Asia',
                       'East Asia', 'Sub-Saharan Africa', 'Sub-Saharan Africa',
                       'Sub-Saharan Africa', 'Sub-Saharan Africa'],
    'income_group'  : ['High', 'High', 'High', 'High',
                       'Upper-Middle', 'Upper-Middle', 'High',
                       'Lower-Middle', 'Lower-Middle', 'Lower-Middle',
                       'Low', 'Low'],
    'gdp_per_capita': [76399.0, 40886.0, 33815.0, 49532.0,
                        9000.5,  9926.0, 32255.0,
                        4163.5,  2065.8,  2363.0,
                         925.1,  1838.0],
    'population'    : [333271341,  67897000, 125124989,   9890400,
                       215313498, 127504125,  51628117,
                        97468029, 218541000,  32395450,
                       123380000,  54027487],
    'poverty_rate'  : [1.2, 0.1, 0.7, 0.0,
                       5.8, 7.3, 0.2,
                       1.5, 39.1, 24.8,
                       26.1, 36.1],
    'growth_rate'   : [2.0, 2.5, 1.5, 3.5,
                       3.0, 2.5, 2.7,
                       6.1, 3.2, 3.8,
                       7.0, 5.2]
}

df = pd.DataFrame(data)
df['log_gdp'] = np.log(df['gdp_per_capita']).round(3)
print('Dataset loaded:', df.shape)
df.head()

Dataset loaded: (12, 8)


,country,region,income_group,gdp_per_capita,population,poverty_rate,growth_rate,log_gdp
0,USA,North America,High,76399.0,333271341,1.2,2.0,11.244
1,France,Europe,High,40886.0,67897000,0.1,2.5,10.619
2,Japan,East Asia,High,33815.0,125124989,0.7,1.5,10.429
3,UAE,Middle East,High,49532.0,9890400,0.0,3.5,10.810
4,Brazil,Latin America,Upper-Middle,9000.5,215313498,5.8,3.0,9.105


---
## Part 1: GroupBy

GroupBy answers questions like: *What is the average poverty rate by region?*

In [2]:
# Average GDP per capita by income group
df.groupby('income_group')['gdp_per_capita'].mean().round(0)

,gdp_per_capita
income_group,
High,46577.0
Low,1382.0
Lower-Middle,2864.0
Upper-Middle,9463.0


In [3]:
# Multiple statistics at once
df.groupby('income_group')['gdp_per_capita'].agg(['mean', 'median', 'std', 'count']).round(0)

,mean,median,std,count
income_group,,,,
High,46577.0,40886.0,18017.0,5
Low,1382.0,1382.0,646.0,2
Lower-Middle,2864.0,2363.0,1135.0,3
Upper-Middle,9463.0,9463.0,654.0,2


In [4]:
# Multiple columns, multiple statistics
df.groupby('region')[['gdp_per_capita', 'poverty_rate', 'growth_rate']].agg(['mean', 'min', 'max']).round(2)

gdp_per_capita                   poverty_rate              \
                             mean      min      max         mean   min   max   
region                                                                         
East Asia                23411.17   4163.5  33815.0         0.80   0.2   1.5   
Europe                   40886.00  40886.0  40886.0         0.10   0.1   0.1   
Latin America             9463.25   9000.5   9926.0         6.55   5.8   7.3   
Middle East              49532.00  49532.0  49532.0         0.00   0.0   0.0   
North America            76399.00  76399.0  76399.0         1.20   1.2   1.2   
Sub-Saharan Africa        1797.98    925.1   2363.0        31.52  24.8  39.1   

                   growth_rate            
                          mean  min  max  
region                                    
East Asia                 3.43  1.5  6.1  
Europe                    2.50  2.5  2.5  
Latin America             2.75  2.5  3.0  
Middle East               3.50  3.5  3.5  
North America             2.00  2.0  2.0  
Sub-Saharan Africa        4.80  3.2  7.0

In [ ]:
# Named aggregations
regional_summary = df.groupby('region').agg(
    n_countries  = ('country',        'count'),
    avg_gdp      = ('gdp_per_capita',  'mean'),
    avg_poverty  = ('poverty_rate',    'mean'),
    avg_growth   = ('growth_rate',     'mean'),
    total_pop_mn = ('population',      lambda x: x.sum() / 1e6)
).round(2)

regional_summary.sort_values('avg_gdp', ascending=False)

In [5]:
# Transform: add group mean as a new column for within-group comparison
df['region_avg_gdp'] = df.groupby('region')['gdp_per_capita'].transform('mean').round(0)
df['gdp_vs_region']  = (df['gdp_per_capita'] - df['region_avg_gdp']).round(0)

df[['country', 'region', 'gdp_per_capita', 'region_avg_gdp', 'gdp_vs_region']].sort_values('region')

,country,region,gdp_per_capita,region_avg_gdp,gdp_vs_region
2,Japan,East Asia,33815.0,23411.0,10404.0
6,South Korea,East Asia,32255.0,23411.0,8844.0
7,Vietnam,East Asia,4163.5,23411.0,-19248.0
1,France,Europe,40886.0,40886.0,0.0
4,Brazil,Latin America,9000.5,9463.0,-462.0
5,Mexico,Latin America,9926.0,9463.0,463.0
3,UAE,Middle East,49532.0,49532.0,0.0
0,USA,North America,76399.0,76399.0,0.0
8,Nigeria,Sub-Saharan Africa,2065.8,1798.0,268.0
9,Ghana,Sub-Saharan Africa,2363.0,1798.0,565.0


---
## Part 2: Merging DataFrames

Real research combines multiple datasets. Pandas `.merge()` works like a SQL JOIN.

In [6]:
# Governance dataset (World Bank Governance Indicators, 2022)
# Rule of law index: -2.5 (weak) to +2.5 (strong)
governance = pd.DataFrame({
    'country'           : ['USA', 'France', 'Japan', 'UAE',
                           'Brazil', 'Mexico', 'South Korea',
                           'Vietnam', 'Nigeria', 'Ghana',
                           'Ethiopia', 'Kenya'],
    'rule_of_law'       : [ 1.55,  1.45,  1.72,  0.52,
                           -0.22, -0.55,  1.05,
                           -0.42, -1.12, -0.18,
                           -0.82, -0.55],
    'govt_effectiveness': [ 1.62,  1.38,  1.73,  1.05,
                            0.02, -0.12,  1.12,
                           -0.02, -1.25,  0.02,
                           -0.85, -0.62]
})

governance.head()

,country,rule_of_law,govt_effectiveness
0,USA,1.55,1.62
1,France,1.45,1.38
2,Japan,1.72,1.73
3,UAE,0.52,1.05
4,Brazil,-0.22,0.02


In [7]:
# Inner join: keep only countries in BOTH datasets
merged = df.merge(governance, on='country', how='inner')
print('Shape after inner merge:', merged.shape)
merged[['country', 'gdp_per_capita', 'rule_of_law', 'govt_effectiveness']].head(6)

Shape after inner merge: (12, 12)


,country,gdp_per_capita,rule_of_law,govt_effectiveness
0,USA,76399.0,1.55,1.62
1,France,40886.0,1.45,1.38
2,Japan,33815.0,1.72,1.73
3,UAE,49532.0,0.52,1.05
4,Brazil,9000.5,-0.22,0.02
5,Mexico,9926.0,-0.55,-0.12


In [8]:
# Left join: keep all rows from left df, fill missing with NaN
governance_partial = governance.iloc[:8]

left_merged = df.merge(governance_partial, on='country', how='left')
print('Shape after left merge:', left_merged.shape)
print('Missing governance data:', left_merged['rule_of_law'].isnull().sum(), 'countries')
left_merged[['country', 'gdp_per_capita', 'rule_of_law']].tail(6)

Shape after left merge: (12, 12)
Missing governance data: 4 countries


,country,gdp_per_capita,rule_of_law
6,South Korea,32255.0,1.05
7,Vietnam,4163.5,-0.42
8,Nigeria,2065.8,NaN
9,Ghana,2363.0,NaN
10,Ethiopia,925.1,NaN
11,Kenya,1838.0,NaN


In [9]:
# Does governance predict income?
corr = merged[['log_gdp', 'rule_of_law', 'govt_effectiveness']].corr().round(3)
print('Correlation matrix:')
print(corr)

Correlation matrix:
                    log_gdp  rule_of_law  govt_effectiveness
log_gdp               1.000        0.880               0.926
rule_of_law           0.880        1.000               0.976
govt_effectiveness    0.926        0.976               1.000


---
## Part 3: Wide vs Long Format

- **Wide**: one row per country, one column per year — common in World Bank downloads
- **Long**: one row per country-year — required for panel regressions

You must know how to convert between them.

In [10]:
# Wide format
wide = pd.DataFrame({
    'country': ['USA', 'France', 'Japan', 'Brazil', 'Vietnam', 'Nigeria', 'Ethiopia', 'Kenya'],
    '2018'   : [62794, 41463, 39290,  9001, 2714, 2026,  772, 1711],
    '2019'   : [65118, 40494, 40247,  8897, 2715, 2097,  856, 1816],
    '2020'   : [63544, 38625, 40113,  7507, 2786, 2097,  936, 1838],
    '2021'   : [70149, 43659, 39340,  7507, 3756, 2085,  925, 2006],
    '2022'   : [76399, 40886, 33815,  9001, 4163, 2066,  925, 1838]
})

print('WIDE FORMAT:')
wide

WIDE FORMAT:


,country,2018,2019,2020,2021,2022
0,USA,62794,65118,63544,70149,76399
1,France,41463,40494,38625,43659,40886
2,Japan,39290,40247,40113,39340,33815
3,Brazil,9001,8897,7507,7507,9001
4,Vietnam,2714,2715,2786,3756,4163
5,Nigeria,2026,2097,2097,2085,2066
6,Ethiopia,772,856,936,925,925
7,Kenya,1711,1816,1838,2006,1838


In [11]:
# Wide to Long using pd.melt()
long = wide.melt(
    id_vars    = ['country'],
    var_name   = 'year',
    value_name = 'gdp_per_capita'
)
long['year'] = long['year'].astype(int)
long = long.sort_values(['country', 'year']).reset_index(drop=True)

print('LONG FORMAT:')
print('Shape:', long.shape)
long.head(12)

LONG FORMAT:
Shape: (40, 3)


,country,year,gdp_per_capita
0,Brazil,2018,9001
1,Brazil,2019,8897
2,Brazil,2020,7507
3,Brazil,2021,7507
4,Brazil,2022,9001
5,Ethiopia,2018,772
6,Ethiopia,2019,856
7,Ethiopia,2020,936
8,Ethiopia,2021,925
9,Ethiopia,2022,925


In [12]:
# Long to Wide using pivot_table()
back_to_wide = long.pivot_table(
    index   = 'country',
    columns = 'year',
    values  = 'gdp_per_capita'
).reset_index()

print('BACK TO WIDE:')
back_to_wide

BACK TO WIDE:


year,country,2018,2019,2020,2021,2022
0,Brazil,9001.0,8897.0,7507.0,7507.0,9001.0
1,Ethiopia,772.0,856.0,936.0,925.0,925.0
2,France,41463.0,40494.0,38625.0,43659.0,40886.0
3,Japan,39290.0,40247.0,40113.0,39340.0,33815.0
4,Kenya,1711.0,1816.0,1838.0,2006.0,1838.0
5,Nigeria,2026.0,2097.0,2097.0,2085.0,2066.0
6,USA,62794.0,65118.0,63544.0,70149.0,76399.0
7,Vietnam,2714.0,2715.0,2786.0,3756.0,4163.0


In [13]:
# Year-on-year GDP growth — easier in long format
long = long.sort_values(['country', 'year'])
long['gdp_growth'] = long.groupby('country')['gdp_per_capita'].pct_change() * 100
long['gdp_growth'] = long['gdp_growth'].round(2)

print('GDP growth rates:')
long[long['country'].isin(['USA', 'Nigeria', 'Vietnam'])].dropna()

GDP growth rates:


,country,year,gdp_per_capita,gdp_growth
26,Nigeria,2019,2097,3.50
27,Nigeria,2020,2097,0.00
28,Nigeria,2021,2085,-0.57
29,Nigeria,2022,2066,-0.91
31,USA,2019,65118,3.70
32,USA,2020,63544,-2.42
33,USA,2021,70149,10.39
34,USA,2022,76399,8.91
36,Vietnam,2019,2715,0.04
37,Vietnam,2020,2786,2.62


---
## Part 4: Cleaning Messy Data

Real-world data is never clean. Every empirical paper spends significant time on this.

In [14]:
# Simulate a messy dataset
messy = pd.DataFrame({
    'Country Name'   : ['  USA', 'france', 'JAPAN', 'Brazil ', 'nigeria', None,    'Kenya'],
    'GDP_per_Capita' : ['76,399', '40886', '33815.0', 'N/A',  '2065.8', '925.1', '1838'],
    'Poverty Rate %' : ['1.2',   '0.1',   '0.7',   '5.8',    '39.1',   '26.1',   None],
    'Year'           : ['2022',  '2022',  '2022',  '2022',   '2022',   '2022',  '2022']
})

print('MESSY DATA:')
messy

MESSY DATA:


,Country Name,GDP_per_Capita,Poverty Rate %,Year
0,USA,"76,399",1.2,2022
1,france,40886,0.1,2022
2,JAPAN,33815.0,0.7,2022
3,Brazil,N/A,5.8,2022
4,nigeria,2065.8,39.1,2022
5,None,925.1,26.1,2022
6,Kenya,1838,None,2022


In [15]:
# Step 1: Fix column names
messy.columns = messy.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('%', 'pct')
print('Cleaned column names:', messy.columns.tolist())

Cleaned column names: ['country_name', 'gdp_per_capita', 'poverty_rate_pct', 'year']


In [16]:
# Step 2: Fix country names
messy['country_name'] = messy['country_name'].str.strip().str.title()
print(messy['country_name'].tolist())

['Usa', 'France', 'Japan', 'Brazil', 'Nigeria', None, 'Kenya']


In [17]:
# Step 3: Fix GDP column — remove commas, replace N/A, convert to float
messy['gdp_per_capita'] = (
    messy['gdp_per_capita']
    .str.replace(',', '')
    .replace('N/A', np.nan)
    .astype(float)
)
print(messy['gdp_per_capita'])

0    76399.0
1    40886.0
2    33815.0
3        NaN
4     2065.8
5      925.1
6     1838.0
Name: gdp_per_capita, dtype: float64


In [18]:
# Step 4: Fix poverty rate
messy['poverty_rate_pct'] = pd.to_numeric(messy['poverty_rate_pct'], errors='coerce')
print(messy['poverty_rate_pct'])

0     1.2
1     0.1
2     0.7
3     5.8
4    39.1
5    26.1
6     NaN
Name: poverty_rate_pct, dtype: float64


In [19]:
# Step 5: Handle missing values
print('Missing values:')
print(messy.isnull().sum())

# Option A: drop rows with any missing
clean_drop = messy.dropna()
print('After dropna():', clean_drop.shape)

# Option B: fill missing GDP with column mean
clean_fill = messy.copy()
clean_fill['gdp_per_capita'] = clean_fill['gdp_per_capita'].fillna(clean_fill['gdp_per_capita'].mean())
print('After fillna(mean) — missing GDP:', clean_fill['gdp_per_capita'].isnull().sum())

Missing values:
country_name        1
gdp_per_capita      1
poverty_rate_pct    1
year                0
dtype: int64
After dropna(): (4, 4)
After fillna(mean) — missing GDP: 0


In [20]:
# Step 6: Fix dtypes and remove duplicates
messy['year'] = messy['year'].astype(int)

messy_with_dupes = pd.concat([messy, messy.iloc[[0]]], ignore_index=True)
print('With duplicate:', messy_with_dupes.shape)
messy_clean = messy_with_dupes.drop_duplicates()
print('After dedup:   ', messy_clean.shape)

With duplicate: (8, 4)
After dedup:    (7, 4)


---
## Part 5: Building a Panel Dataset from World Bank

In [21]:
!pip install wbgapi -q

In [22]:
import wbgapi as wb

iso_codes = ['USA', 'FRA', 'JPN', 'ARE',
             'BRA', 'MEX', 'KOR',
             'VNM', 'NGA', 'GHA',
             'ETH', 'KEN']

indicators = {
    'NY.GDP.PCAP.CD' : 'gdp_per_capita',
    'SI.POV.DDAY'    : 'poverty_rate',
    'SP.POP.TOTL'    : 'population',
    'SE.ADT.LITR.ZS' : 'literacy_rate'
}

frames = []
for code, name in indicators.items():
    temp = wb.data.DataFrame(code,
                              economy=iso_codes,
                              time=range(2010, 2023),
                              labels=True).reset_index()
    temp = temp.melt(id_vars=['economy', 'Country'],
                     var_name='year',
                     value_name=name)
    temp['year'] = temp['year'].str.replace('YR', '').astype(int)
    temp = temp.rename(columns={'economy': 'iso_code', 'Country': 'country'})
    frames.append(temp)

panel = frames[0]
for f in frames[1:]:
    panel = panel.merge(f[['iso_code', 'year', f.columns[-1]]],
                        on=['iso_code', 'year'], how='left')

panel = panel.sort_values(['country', 'year']).reset_index(drop=True)
panel['log_gdp']    = np.log(panel['gdp_per_capita'])
panel['gdp_growth'] = panel.groupby('country')['gdp_per_capita'].pct_change() * 100

print('Panel shape:', panel.shape)
panel.head(10)

Panel shape: (156, 9)


,iso_code,country,year,gdp_per_capita,poverty_rate,population,literacy_rate,log_gdp,gdp_growth
0,BRA,Brazil,2010,11403.284004,NaN,193701929.0,90.379997,9.341657,NaN
1,BRA,Brazil,2011,13396.626316,7.1,195284734.0,91.410004,9.502758,17.480423
2,BRA,Brazil,2012,12521.723845,6.4,196876111.0,91.339996,9.435220,-6.530767
3,BRA,Brazil,2013,12458.890340,5.4,198478299.0,91.480003,9.430190,-0.501796
4,BRA,Brazil,2014,12274.994163,4.8,200085127.0,91.730003,9.415319,-1.476024
5,BRA,Brazil,2015,8936.195589,5.4,201675532.0,92.050003,9.097865,-27.200001
6,BRA,Brazil,2016,8836.285460,6.5,203218114.0,92.809998,9.086622,-1.118039
7,BRA,Brazil,2017,10080.507872,7.0,204703445.0,93.080002,9.218359,14.080831
8,BRA,Brazil,2018,9300.660729,6.9,206107261.0,93.230003,9.137841,-7.736189
9,BRA,Brazil,2019,9029.833044,7.0,207455459.0,93.892627,9.108289,-2.911919


In [23]:
print('Missing values per column:')
print(panel.isnull().sum())
print()
print('Total observations:', len(panel))
print('Complete rows:     ', panel.dropna().shape[0])

Missing values per column:
iso_code            0
country             0
year                0
gdp_per_capita      0
poverty_rate       68
population          0
literacy_rate     114
log_gdp             0
gdp_growth         12
dtype: int64

Total observations: 156
Complete rows:      20


In [24]:
panel.to_csv('panel_data.csv', index=False)
print('Saved to panel_data.csv')

Saved to panel_data.csv


---
## Summary

| Task | Method |
|------|--------|
| Group and aggregate | `df.groupby('col').agg(...)` |
| Within-group stat | `df.groupby('col')['x'].transform('mean')` |
| Merge datasets | `df.merge(df2, on='key', how='inner/left')` |
| Wide to Long | `pd.melt(df, id_vars=..., var_name=..., value_name=...)` |
| Long to Wide | `df.pivot_table(index=..., columns=..., values=...)` |
| Year-on-year change | `.groupby('country')['col'].pct_change()` |
| Fix column names | `.str.strip().str.lower().str.replace(...)` |
| Handle missing | `.dropna()`, `.fillna()`, `pd.to_numeric(..., errors='coerce')` |
| Remove duplicates | `.drop_duplicates()` |

**Next lecture:** Data Visualization — matplotlib and seaborn.

---
*ECON N171 | Summer 2026*